# Canny Edge Detection: Essential Validation & Enhancement Study

This notebook evaluates the **4 essential configurations** of the Canny edge detection pipeline on the **139 validation PCB images** (`data/dataset_split.csv`).

### Enhancement Strategy:
1. **Baseline Failure**: Raw edge differencing produces thousands of discrete single-pixel edge fragments (Precision < 0.1%, IoU < 0.10 against filled annotation boxes).
2. **Morphological Closing (`cv2.MORPH_CLOSE`)**: Bridges fragmented edge contours into solid polygon regions.
3. **Minimum Area Filtering (`min_area`)**: Eliminates high-frequency trace edge noise.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import yaml

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == 'validation' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from algorithms.common import load_image, preprocess_pair
from algorithms.evaluation import evaluate_boxes, parse_voc_boxes
from algorithms.canny import detect_canny
from algorithms.preprocessing import build_preprocessing_config

plt.rcParams['figure.dpi'] = 110
print(f'Project Root: {PROJECT_ROOT}')

# Smart auto-run fallback: auto-compute benchmark if outputs/metrics CSV is missing
res_csv = PROJECT_ROOT / 'outputs' / 'metrics' / 'essential_validation_comparison.csv'
if not res_csv.exists():
    print('⚠️ Validation metrics CSV not found. Auto-running validation benchmark on 139 validation images...')
    from scripts.evaluation.run_essential_validation import main as run_benchmark
    run_benchmark()
    print('✅ Benchmark successfully generated!')


## 1. Essential Validation Results (4 Core Configurations)

In [ ]:
df = pd.read_csv(res_csv)
canny_df = df[df['algorithm'] == 'Canny'][['combination_id', 'description', 'precision', 'recall', 'f1_score', 'mean_runtime_ms']]
display(canny_df)


## 2. Visual Comparison of Canny Configurations

In [ ]:
plt.figure(figsize=(9, 4))
plt.barh(canny_df['combination_id'], canny_df['f1_score'], color='#e74c3c')
plt.xlabel('Validation F1-Score (IoU >= 0.50)')
plt.title('Canny: F1-Score across 4 Essential Configurations')
plt.xlim(0, 0.10)
for i, v in enumerate(canny_df['f1_score']):
    plt.text(v + 0.002, i, f'{v:.4f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Visual Detection Overlay on Defect Samples

In [ ]:
with open(PROJECT_ROOT / 'configs' / 'frozen_parameters.yaml') as f:
    cfg = yaml.safe_load(f)
prep_cfg = build_preprocessing_config(cfg.get('preprocessing'))

manifest = pd.read_csv(PROJECT_ROOT / 'data' / 'dataset_split.csv')
val_subset = manifest[manifest['split'] == 'validation']
sample_row = val_subset[val_subset['defect_class'] == 'spur'].iloc[0]

ref = load_image(PROJECT_ROOT / sample_row['reference_path'])
def_img = load_image(PROJECT_ROOT / sample_row['image_path'])
gt_boxes = parse_voc_boxes(PROJECT_ROOT / sample_row['annotation_path'])

det = detect_canny(ref, def_img, low_threshold=50.0, high_threshold=150.0, morph_close=5, min_area=300.0, preprocessing_config=prep_cfg)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(ref, cv2.COLOR_BGR2RGB))
axes[0].set_title('Reference PCB')
axes[0].axis('off')

axes[1].imshow(det.edge_difference, cmap='gray')
axes[1].set_title('Edge Difference Map')
axes[1].axis('off')

overlay = cv2.cvtColor(def_img.copy(), cv2.COLOR_BGR2RGB)
for b in gt_boxes:
    cv2.rectangle(overlay, (int(b['xmin']), int(b['ymin'])), (int(b['xmax']), int(b['ymax'])), (0, 255, 0), 4)
for b in det.boxes:
    cv2.rectangle(overlay, (int(b['xmin']), int(b['ymin'])), (int(b['xmax']), int(b['ymax'])), (255, 0, 0), 3)
axes[2].imshow(overlay)
axes[2].set_title(f'Predictions (Red: {len(det.boxes)}) vs Ground Truth (Green: {len(gt_boxes)})')
axes[2].axis('off')
plt.tight_layout()
plt.show()
